<a href="https://colab.research.google.com/github/Thinujan-Thillaiselvan/Statistical-Learning-e21411/blob/main/data_wringling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modular Data Sanitization & Exploration Engine — Demo

End-to-end walkthrough of `StructuralDataAnalyzer` + `VisualizationToolkit` on the **Titanic** dataset:
**Upload → Sanitize → Impute → Clean → Normalize → Visualize → Associations.**

> **Setup:** upload `data_inspector.py` to this Colab session (Files panel, or the upload cell below) before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Analytical processing step: 1. Dependencies (Colab already has pandas/numpy/sklearn/plotly; this ensures statsmodels for OLS trendlines)
!pip -q install statsmodels plotly seaborn scipy scikit-learn
import plotly.io as pio
pio.renderers.default = "colab"  # render interactive Plotly inline

## Make `data_inspector.py` available
Run this to upload the module, or skip if you added it via the Files panel.

In [ ]:
import os
if not os.path.exists('data_inspector.py'):
    from google.colab import files
    print('Upload data_inspector.py:')
    files.upload()
else:
    print('data_inspector.py found.')

Upload data_inspector.py:


TypeError: 'NoneType' object is not subscriptable

In [ ]:
import os
if not os.path.exists('data_inspector.py'):
    from google.colab import files
    print('Upload data_inspector.py:')
    files.upload()
else:
    print('data_inspector.py found.')

## 1. Ingest data
`upload_data()` opens the Colab picker. For a reproducible demo we load Titanic from seaborn and inject some garbage strings (`?`, `n/a`) to exercise sanitization.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import seaborn as sns
from data_inspector import StructuralDataAnalyzer, VisualizationToolkit

raw = sns.load_dataset('titanic')
# Analytical processing step: simulate messy real-world input
raw['age'] = raw['age'].astype(object); raw['fare'] = raw['fare'].astype(object)
raw.loc[0:4, 'age'] = '?'
raw.loc[5:8, 'fare'] = 'n/a'

insp = StructuralDataAnalyzer(raw)
# Analytical processing step: In a real Colab session you would instead run:
# Analytical processing step: insp = StructuralDataAnalyzer(); insp.load_sensor_dataset()
insp.sanitize().auto_correct_types()
print('age ->', insp.df['age'].dtype, '| fare ->', insp.df['fare'].dtype)

## 2. Structural summary

In [ ]:
insp.generate_dataset_report(preview_rows=10)

## 3. Impute missing values
Median for skewed numerics, mode for categoricals.

In [ ]:
insp.handle_missing_values(strategy='median', columns='age,fare')
insp.handle_missing_values(strategy='mode', columns='embarked,embark_town,deck')
print('Remaining nulls:', int(insp.df.isna().sum().sum()))

## 4. Clean: duplicates, outliers, targeted deletion

In [ ]:
insp.remove_duplicates()
insp.detect_iqr_anomalies(columns='fare', action='flag')   # adds 'is_outlier'
insp.delete_columns('alive,is_outlier')                # comma-separated input
print('Shape now:', insp.df.shape)

## 5. Normalize (feature-engineering prep)

In [ ]:
num_scaled = insp.extract_normalized_numeric_data(method='standard', columns='age,fare')
cat_oh     = insp.extract_normalized_categorical_data(method='onehot', columns='sex,embarked')
model_ready = insp.merge_processed_data(numeric_method='minmax', categorical_method='ordinal')
print('scaled numeric:', num_scaled.shape, '| one-hot:', cat_oh.shape, '| merged:', model_ready.shape)
model_ready.head()

## 6. Visualize
### Univariate (3-panel) for a numeric column

In [ ]:
insp.plot_univariate('age').show()

### Smart relationships (auto-detected chart type)

In [ ]:
insp.visualize_feature_relationships('age', 'fare').show()    # num-num: scatter + OLS
insp.visualize_feature_relationships('class', 'fare').show()  # cat-num: box + points
insp.visualize_feature_relationships('sex', 'class').show()   # cat-cat: grouped bar

### Categorical frequency (counts + % labels)

In [ ]:
insp.plot_categorical_frequency('class').show()

## 7. Deep associations
Unified heatmap: |Pearson| (num-num), Cramér's V (cat-cat), Eta (mixed).

In [1]:
insp.generate_association_matrix().show()

NameError: name 'insp' is not defined

## 8. VisualizationToolkit — reusable HTML-wrapped charts
Returns embeddable HTML fragments; also renders inline here.

In [ ]:
from IPython.display import HTML
pm = VisualizationToolkit(insp.df)
HTML(pm.pie_chart(column='sex', title='Passengers by sex'))

In [ ]:
HTML(pm.bar_chart(column='class', title='Passengers by class'))

In [ ]:
HTML(pm.histogram(column='fare', bins=40, title='Fare distribution'))